# View 720 80pct

В этом ноутбуке исследуем две близкие метрики просмотров: `Просмотрено 80% ур или видеоконт` для `module = 1` и `Просмотрено 720ед видеоконт и 80% ур` для `module = 2`. Цель — понять, насколько их можно предсказывать по raw LMS-данным без использования `stats__module_*` как источника признаков.


## План

1. Зафиксировать единицу наблюдения: одна строка — один `user-course`.
2. Взять обе метрики из [target_logic_df.csv](/Users/maria/Desktop/uni/hackathon/target_logic_df.csv): мягкий аналог для `module = 1` и основной критерий для `module = 2`.
3. Импортировать raw-таблицы, которые могут дать признаки просмотров: медиа-сессии, уроки пользователя, действия на курсе, состояние курса и пользовательские признаки.
4. Восстановить самые прямые raw-признаки просмотров на уровне `user-course`.
5. Проверить, насколько эти прямые сигналы уже объясняют обе метрики.
6. После этого собрать более широкие `user-course` и `user` признаки и перейти к `model_df`.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)

BASE_DIR = Path("/Users/maria/Desktop/uni/hackathon")
SRC_DIR = BASE_DIR / "src"


## Target

Импортируем объединённую таблицу таргетов и оставляем две разные целевые переменные. Для `module = 1` берём близкий аналог `Просмотрено 80% ур или видеоконт`, для `module = 2` — основной критерий `Просмотрено 720ед видеоконт и 80% ур`.


In [2]:
target_logic_df = pd.read_csv(
    BASE_DIR / "target_logic_df.csv",
    parse_dates=["enrolled_at", "pa_submitted_at"],
    low_memory=False,
)

target_logic_df["module"] = pd.to_numeric(target_logic_df["module"], errors="coerce").astype("Int8")
target_logic_df["user_id"] = pd.to_numeric(target_logic_df["user_id"], errors="coerce").astype("Int32")
target_logic_df["course_id"] = pd.to_numeric(target_logic_df["course_id"], errors="coerce").astype("Int32")
target_logic_df["teacher_id"] = pd.to_numeric(target_logic_df["teacher_id"], errors="coerce").astype("Int32")
target_logic_df["group_template_id"] = pd.to_numeric(target_logic_df["group_template_id"], errors="coerce").astype("Int32")

module_1_metric_df = target_logic_df[target_logic_df["module"] == 1].copy()
module_1_metric_df["metric_name"] = "Просмотрено 80% ур или видеоконт"
module_1_metric_df["target"] = (module_1_metric_df["content_threshold_status"] == "Да").astype("Int8")

module_2_metric_df = target_logic_df[target_logic_df["module"] == 2].copy()
module_2_metric_df["metric_name"] = "Просмотрено 720ед видеоконт и 80% ур"
module_2_metric_df["target"] = (module_2_metric_df["content_threshold_status"] == "Да").astype("Int8")

view_target_df = pd.concat([module_1_metric_df, module_2_metric_df], ignore_index=True)
view_target_df = view_target_df[
    [
        "module",
        "metric_name",
        "user_id",
        "course_id",
        "teacher_id",
        "group_template_id",
        "level",
        "course_name",
        "enrolled_at",
        "content_threshold_status",
        "target_status",
        "target",
    ]
].copy()

target_summary_df = pd.DataFrame(
    {
        "module": [1, 2, "all"],
        "metric_name": [
            "Просмотрено 80% ур или видеоконт",
            "Просмотрено 720ед видеоконт и 80% ур",
            "both",
        ],
        "rows": [
            module_1_metric_df.shape[0],
            module_2_metric_df.shape[0],
            view_target_df.shape[0],
        ],
        "target_1_rows": [
            int(module_1_metric_df["target"].sum()),
            int(module_2_metric_df["target"].sum()),
            int(view_target_df["target"].sum()),
        ],
        "target_0_rows": [
            int((module_1_metric_df["target"] == 0).sum()),
            int((module_2_metric_df["target"] == 0).sum()),
            int((view_target_df["target"] == 0).sum()),
        ],
        "target_1_pct": [
            round(module_1_metric_df["target"].mean() * 100, 2),
            round(module_2_metric_df["target"].mean() * 100, 2),
            round(view_target_df["target"].mean() * 100, 2),
        ],
    }
)

display(view_target_df.head())
display(target_summary_df)


,module,metric_name,user_id,course_id,teacher_id,group_template_id,level,course_name,enrolled_at,content_threshold_status,target_status,target
0,1,Просмотрено 80% ур или видеоконт,701741,1029,699869,1149,Начальный,Python. Начальный уровень. Онлайн. 1 модуль,2025-09-19,Да,Завершил,1
1,1,Просмотрено 80% ур или видеоконт,737977,1029,730341,1216,Начальный,Python. Начальный уровень. Онлайн. 1 модуль,2025-11-05,Да,Завершил,1
2,1,Просмотрено 80% ур или видеоконт,722851,1030,730208,1186,Базовый,Python. Базовый уровень. Онлайн. 1 модуль,2025-10-21,Нет,Отчислен,0
3,1,Просмотрено 80% ур или видеоконт,709724,1030,700089,1108,Базовый,Python. Базовый уровень. Онлайн. 1 модуль,2025-09-23,Да,Завершил,1
4,1,Просмотрено 80% ур или видеоконт,717763,1030,718519,1055,Базовый,Python. Базовый уровень. Онлайн. 1 модуль,2025-10-09,Нет,Отчислен,0


,module,metric_name,rows,target_1_rows,target_0_rows,target_1_pct
0,1,Просмотрено 80% ур или видеоконт,3261,2101,1160,64.43
1,2,Просмотрено 720ед видеоконт и 80% ур,1955,1833,122,93.76
2,all,both,5216,3934,1282,75.42


## Импорт данных

Импортируем только raw-таблицы, которые нужны для анализа и предсказания этой метрики. `stats__module_*` сюда не включаем.


In [3]:
users_df = pd.read_csv(
    SRC_DIR / "users.csv",
    usecols=["id", "created_at", "sign_in_count", "grade_id", "subscribed", "xp", "wk_gender", "timezone"],
    thousands=",",
    low_memory=False,
)
users_df["id"] = pd.to_numeric(users_df["id"], errors="coerce").astype("Int32")
users_df["created_at"] = pd.to_datetime(users_df["created_at"], errors="coerce")
users_df["subscribed"] = users_df["subscribed"].astype("boolean")
users_df["grade_id"] = pd.to_numeric(users_df["grade_id"], errors="coerce").astype("Int32")
users_df["sign_in_count"] = pd.to_numeric(users_df["sign_in_count"], errors="coerce").astype("Int32")
users_df["xp"] = pd.to_numeric(users_df["xp"], errors="coerce").astype("Int32")
users_df["wk_gender"] = pd.to_numeric(users_df["wk_gender"], errors="coerce").astype("Int8")

users_courses_df = pd.read_csv(
    SRC_DIR / "users_courses.csv",
    usecols=[
        "id",
        "user_id",
        "course_id",
        "state",
        "created_at",
        "updated_at",
        "access_finished_at",
        "wk_points",
        "wk_max_points",
        "wk_max_viewable_lessons",
        "wk_max_task_count",
        "wk_officially_started_at",
        "wk_course_completed_at",
    ],
    thousands=",",
    low_memory=False,
)
users_courses_df["id"] = pd.to_numeric(users_courses_df["id"], errors="coerce").astype("Int32")
users_courses_df["user_id"] = pd.to_numeric(users_courses_df["user_id"], errors="coerce").astype("Int32")
users_courses_df["course_id"] = pd.to_numeric(users_courses_df["course_id"], errors="coerce").astype("Int32")
users_courses_df["created_at"] = pd.to_datetime(users_courses_df["created_at"], errors="coerce")
users_courses_df["updated_at"] = pd.to_datetime(users_courses_df["updated_at"], errors="coerce")
users_courses_df["access_finished_at"] = pd.to_datetime(users_courses_df["access_finished_at"], errors="coerce")
users_courses_df["wk_officially_started_at"] = pd.to_datetime(users_courses_df["wk_officially_started_at"], errors="coerce")
users_courses_df["wk_course_completed_at"] = pd.to_datetime(users_courses_df["wk_course_completed_at"], errors="coerce")
users_courses_df["state"] = users_courses_df["state"].astype("category")

lessons_df = pd.read_csv(
    SRC_DIR / "lessons.csv",
    usecols=[
        "id",
        "course_id",
        "lesson_number",
        "conspect_expected",
        "task_expected",
        "wk_max_points",
        "wk_task_count",
        "wk_video_duration",
        "wk_attendance_tracking_enabled",
    ],
    thousands=",",
    low_memory=False,
)
lessons_df["id"] = pd.to_numeric(lessons_df["id"], errors="coerce").astype("Int32")
lessons_df["course_id"] = pd.to_numeric(lessons_df["course_id"], errors="coerce").astype("Int32")

groups_df = pd.read_csv(
    SRC_DIR / "groups.csv",
    usecols=[
        "id",
        "lesson_id",
        "starts_at",
        "duration",
        "state",
        "group_template_id",
        "video_available",
        "wk_actual_started_at",
        "wk_actual_finished_at",
    ],
    thousands=",",
    low_memory=False,
)
groups_df["id"] = pd.to_numeric(groups_df["id"], errors="coerce").astype("Int32")
groups_df["lesson_id"] = pd.to_numeric(groups_df["lesson_id"], errors="coerce").astype("Int32")
groups_df["group_template_id"] = pd.to_numeric(groups_df["group_template_id"], errors="coerce").astype("Int32")
groups_df["starts_at"] = pd.to_datetime(groups_df["starts_at"], errors="coerce")
groups_df["wk_actual_started_at"] = pd.to_datetime(groups_df["wk_actual_started_at"], errors="coerce")
groups_df["wk_actual_finished_at"] = pd.to_datetime(groups_df["wk_actual_finished_at"], errors="coerce")
groups_df["state"] = groups_df["state"].astype("category")
groups_df["video_available"] = groups_df["video_available"].astype("boolean")

user_access_histories_df = pd.read_csv(
    SRC_DIR / "user_access_histories.csv",
    usecols=["users_course_id", "access_started_at", "access_expired_at", "activator_class"],
    thousands=",",
    low_memory=False,
)
user_access_histories_df["users_course_id"] = pd.to_numeric(user_access_histories_df["users_course_id"], errors="coerce").astype("Int32")
user_access_histories_df["access_started_at"] = pd.to_datetime(user_access_histories_df["access_started_at"], errors="coerce")
user_access_histories_df["access_expired_at"] = pd.to_datetime(user_access_histories_df["access_expired_at"], errors="coerce")
user_access_histories_df["activator_class"] = user_access_histories_df["activator_class"].astype("category")

user_lessons_df = pd.read_csv(
    SRC_DIR / "user_lessons.csv",
    usecols=[
        "user_id",
        "lesson_id",
        "group_id",
        "video_visited",
        "translation_visited",
        "users_course_id",
        "solved",
        "solved_tasks_count",
        "wk_points",
        "wk_solved_task_count",
    ],
    thousands=",",
    low_memory=False,
)
user_lessons_df["user_id"] = pd.to_numeric(user_lessons_df["user_id"], errors="coerce").astype("Int32")
user_lessons_df["lesson_id"] = pd.to_numeric(user_lessons_df["lesson_id"], errors="coerce").astype("Int32")
user_lessons_df["group_id"] = pd.to_numeric(user_lessons_df["group_id"], errors="coerce").astype("Int32")
user_lessons_df["users_course_id"] = pd.to_numeric(user_lessons_df["users_course_id"], errors="coerce").astype("Int32")
user_lessons_df["video_visited"] = user_lessons_df["video_visited"].astype("boolean")
user_lessons_df["translation_visited"] = user_lessons_df["translation_visited"].astype("boolean")
user_lessons_df["solved"] = user_lessons_df["solved"].astype("boolean")

wk_media_view_sessions_df = pd.read_csv(
    SRC_DIR / "wk_media_view_sessions.csv",
    usecols=["resource_type", "resource_id", "viewer_id", "segments_total", "viewed_segments_count", "started_at", "kind"],
    thousands=",",
    low_memory=False,
)
wk_media_view_sessions_df["resource_id"] = pd.to_numeric(wk_media_view_sessions_df["resource_id"], errors="coerce").astype("Int32")
wk_media_view_sessions_df["viewer_id"] = pd.to_numeric(wk_media_view_sessions_df["viewer_id"], errors="coerce").astype("Int32")
wk_media_view_sessions_df["started_at"] = pd.to_datetime(wk_media_view_sessions_df["started_at"], errors="coerce")
wk_media_view_sessions_df["resource_type"] = wk_media_view_sessions_df["resource_type"].astype("category")
wk_media_view_sessions_df["kind"] = wk_media_view_sessions_df["kind"].astype("category")

wk_users_courses_actions_df = pd.read_csv(
    SRC_DIR / "wk_users_courses_actions.csv",
    usecols=["user_id", "users_course_id", "action", "created_at", "updated_at", "lesson_id"],
    thousands=",",
    low_memory=False,
)
wk_users_courses_actions_df["user_id"] = pd.to_numeric(wk_users_courses_actions_df["user_id"], errors="coerce").astype("Int32")
wk_users_courses_actions_df["users_course_id"] = pd.to_numeric(wk_users_courses_actions_df["users_course_id"], errors="coerce").astype("Int32")
wk_users_courses_actions_df["lesson_id"] = pd.to_numeric(wk_users_courses_actions_df["lesson_id"], errors="coerce").astype("Int32")
wk_users_courses_actions_df["created_at"] = pd.to_datetime(wk_users_courses_actions_df["created_at"], errors="coerce")
wk_users_courses_actions_df["updated_at"] = pd.to_datetime(wk_users_courses_actions_df["updated_at"], errors="coerce")
wk_users_courses_actions_df["action"] = wk_users_courses_actions_df["action"].astype("category")

user_answers_df = pd.read_csv(
    SRC_DIR / "user_answers.csv",
    usecols=["user_id", "task_id", "attempts", "solved", "points", "max_attempts", "resource_type", "resource_id", "submitted_at"],
    thousands=",",
    low_memory=False,
)
user_answers_df["user_id"] = pd.to_numeric(user_answers_df["user_id"], errors="coerce").astype("Int32")
user_answers_df["task_id"] = pd.to_numeric(user_answers_df["task_id"], errors="coerce").astype("Int32")
user_answers_df["resource_id"] = pd.to_numeric(user_answers_df["resource_id"], errors="coerce").astype("Int32")
user_answers_df["submitted_at"] = pd.to_datetime(user_answers_df["submitted_at"], errors="coerce")
user_answers_df["solved"] = user_answers_df["solved"].astype("boolean")
user_answers_df["resource_type"] = user_answers_df["resource_type"].astype("category")

user_trainings_df = pd.read_csv(
    SRC_DIR / "user_trainings.csv",
    usecols=["user_id", "training_id", "solved_tasks_count", "earned_points", "type", "state", "submitted_answers_count", "started_at", "finished_at", "mark"],
    thousands=",",
    low_memory=False,
)
user_trainings_df["user_id"] = pd.to_numeric(user_trainings_df["user_id"], errors="coerce").astype("Int32")
user_trainings_df["training_id"] = pd.to_numeric(user_trainings_df["training_id"], errors="coerce").astype("Int32")
user_trainings_df["started_at"] = pd.to_datetime(user_trainings_df["started_at"], errors="coerce")
user_trainings_df["finished_at"] = pd.to_datetime(user_trainings_df["finished_at"], errors="coerce")
user_trainings_df["type"] = user_trainings_df["type"].astype("category")
user_trainings_df["state"] = user_trainings_df["state"].astype("category")

dataframes = {
    "view_target_df": view_target_df,
    "users_df": users_df,
    "users_courses_df": users_courses_df,
    "lessons_df": lessons_df,
    "groups_df": groups_df,
    "user_access_histories_df": user_access_histories_df,
    "user_lessons_df": user_lessons_df,
    "wk_media_view_sessions_df": wk_media_view_sessions_df,
    "wk_users_courses_actions_df": wk_users_courses_actions_df,
    "user_answers_df": user_answers_df,
    "user_trainings_df": user_trainings_df,
}

import_summary_df = pd.DataFrame(
    {
        "dataframe": list(dataframes.keys()),
        "rows": [df.shape[0] for df in dataframes.values()],
        "cols": [df.shape[1] for df in dataframes.values()],
        "memory_mb": [round(df.memory_usage(deep=True).sum() / 1024**2, 2) for df in dataframes.values()],
    }
)

display(import_summary_df)


KeyboardInterrupt: 

## Прямые raw-признаки просмотров

Сначала собираем самые прямые сигналы просмотров на уровне `user-course`: сколько уроков пользователь открыл, сколько медиа-сессий у него было, сколько сегментов он просмотрел и какую долю уроков курса он реально затронул.


In [ ]:
view_target_unique_df = view_target_df.drop_duplicates(subset=["module", "user_id", "course_id"]).copy()

user_course_base_df = view_target_unique_df.merge(
    users_courses_df[["id", "user_id", "course_id", "state", "created_at", "access_finished_at", "wk_officially_started_at"]],
    on=["user_id", "course_id"],
    how="left",
)
user_course_base_df = user_course_base_df.rename(columns={"id": "users_course_id", "created_at": "users_course_created_at"})

course_view_df = lessons_df.groupby("course_id", dropna=False).agg(
    total_lessons=("id", "nunique"),
    total_video_duration=("wk_video_duration", "sum"),
).reset_index()

user_lessons_df["lesson_viewed"] = user_lessons_df["video_visited"].fillna(False) | user_lessons_df["translation_visited"].fillna(False)
user_lesson_view_df = user_lessons_df.groupby("users_course_id", dropna=False).agg(
    user_lesson_rows=("lesson_id", "size"),
    lessons_viewed_raw=("lesson_viewed", "sum"),
    video_visited_lessons=("video_visited", "sum"),
    translation_visited_lessons=("translation_visited", "sum"),
).reset_index()

lesson_media_df = wk_media_view_sessions_df[wk_media_view_sessions_df["resource_type"] == "Lesson"].copy()
lesson_media_df = lesson_media_df.merge(
    lessons_df[["id", "course_id"]],
    left_on="resource_id",
    right_on="id",
    how="left",
)
lesson_media_view_df = lesson_media_df.groupby(["viewer_id", "course_id"], dropna=False).agg(
    lesson_media_sessions=("resource_id", "size"),
    lesson_media_resources=("resource_id", "nunique"),
    lesson_media_segments_total=("segments_total", "sum"),
    lesson_media_segments_viewed=("viewed_segments_count", "sum"),
).reset_index()

group_media_df = wk_media_view_sessions_df[wk_media_view_sessions_df["resource_type"] == "Group"].copy()
group_media_df = group_media_df.merge(
    groups_df[["id", "lesson_id"]],
    left_on="resource_id",
    right_on="id",
    how="left",
)
group_media_df = group_media_df.merge(
    lessons_df[["id", "course_id"]],
    left_on="lesson_id",
    right_on="id",
    how="left",
)
group_media_view_df = group_media_df.groupby(["viewer_id", "course_id"], dropna=False).agg(
    group_media_sessions=("resource_id", "size"),
    group_media_resources=("resource_id", "nunique"),
    group_media_segments_total=("segments_total", "sum"),
    group_media_segments_viewed=("viewed_segments_count", "sum"),
).reset_index()

direct_view_features_df = user_course_base_df.merge(course_view_df, on="course_id", how="left")
direct_view_features_df = direct_view_features_df.merge(user_lesson_view_df, on="users_course_id", how="left")
direct_view_features_df = direct_view_features_df.merge(
    lesson_media_view_df,
    left_on=["user_id", "course_id"],
    right_on=["viewer_id", "course_id"],
    how="left",
)
direct_view_features_df = direct_view_features_df.drop(columns=["viewer_id"])
direct_view_features_df = direct_view_features_df.merge(
    group_media_view_df,
    left_on=["user_id", "course_id"],
    right_on=["viewer_id", "course_id"],
    how="left",
)
direct_view_features_df = direct_view_features_df.drop(columns=["viewer_id"])

direct_view_features_df["lessons_viewed_share_raw"] = direct_view_features_df["lessons_viewed_raw"] / direct_view_features_df["total_lessons"]
direct_view_features_df["lesson_media_share_raw"] = direct_view_features_df["lesson_media_segments_viewed"] / direct_view_features_df["lesson_media_segments_total"]
direct_view_features_df["group_media_share_raw"] = direct_view_features_df["group_media_segments_viewed"] / direct_view_features_df["group_media_segments_total"]

display(direct_view_features_df.head())


,module,metric_name,user_id,course_id,teacher_id,group_template_id,level,course_name,enrolled_at,content_threshold_status,target_status,target,users_course_id,state,users_course_created_at,access_finished_at,wk_officially_started_at,total_lessons,total_video_duration,user_lesson_rows,lessons_viewed_raw,video_visited_lessons,translation_visited_lessons,lesson_media_sessions,lesson_media_resources,lesson_media_segments_total,lesson_media_segments_viewed,group_media_sessions,group_media_resources,group_media_segments_total,group_media_segments_viewed,lessons_viewed_share_raw,lesson_media_share_raw,group_media_share_raw
0,1,Просмотрено 80% ур или видеоконт,701741,1029,699869,1149,Начальный,Python. Начальный уровень. Онлайн. 1 модуль,2025-09-19,Да,Завершил,1,588595,active,2025-10-28 17:22:00,2026-04-28,NaT,23.0,0.0,23,20,20,0,<NA>,NaN,NaN,NaN,23,20.0,1029.0,894.0,0.869565,NaN,0.868805
1,1,Просмотрено 80% ур или видеоконт,737977,1029,730341,1216,Начальный,Python. Начальный уровень. Онлайн. 1 модуль,2025-11-05,Да,Завершил,1,623160,active,2025-11-13 16:40:00,2026-05-13,NaT,23.0,0.0,23,20,13,19,<NA>,NaN,NaN,NaN,30,20.0,1378.0,900.0,0.869565,NaN,0.653120
2,1,Просмотрено 80% ур или видеоконт,722851,1030,730208,1186,Базовый,Python. Базовый уровень. Онлайн. 1 модуль,2025-10-21,Нет,Отчислен,0,602174,active,2025-11-07 08:23:00,2026-05-07,NaT,23.0,0.0,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN
3,1,Просмотрено 80% ур или видеоконт,709724,1030,700089,1108,Базовый,Python. Базовый уровень. Онлайн. 1 модуль,2025-09-23,Да,Завершил,1,573746,active,2025-10-15 21:09:00,2026-04-16,NaT,23.0,0.0,23,20,18,19,<NA>,NaN,NaN,NaN,35,20.0,1725.0,979.0,0.869565,NaN,0.567536
4,1,Просмотрено 80% ур или видеоконт,717763,1030,718519,1055,Базовый,Python. Базовый уровень. Онлайн. 1 модуль,2025-10-09,Нет,Отчислен,0,568668,active,2025-10-13 18:42:00,2026-04-13,NaT,23.0,0.0,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN


## Покрытие прямых сигналов

Перед сравнением с таргетом полезно понять, насколько вообще заполняются прямые raw-признаки. Здесь смотрим, для какой доли строк находятся `users_course_id`, уроки пользователя и медиа-сессии.


In [ ]:
direct_view_features_df["total_media_segments_total"] = direct_view_features_df["lesson_media_segments_total"].fillna(0) + direct_view_features_df["group_media_segments_total"].fillna(0)
direct_view_features_df["total_media_segments_viewed"] = direct_view_features_df["lesson_media_segments_viewed"].fillna(0) + direct_view_features_df["group_media_segments_viewed"].fillna(0)
direct_view_features_df["raw_lessons_80_status"] = (direct_view_features_df["lessons_viewed_share_raw"] >= 0.8).fillna(False).astype("Int8")
direct_view_features_df["raw_media_720_status"] = (direct_view_features_df["total_media_segments_viewed"] >= 720).fillna(False).astype("Int8")

coverage_summary_df = pd.DataFrame(
    {
        "metric": [
            "rows",
            "users_course_id_found",
            "user_lessons_found",
            "lesson_media_found",
            "group_media_found",
        ],
        "rows": [
            direct_view_features_df.shape[0],
            int(direct_view_features_df["users_course_id"].notna().sum()),
            int(direct_view_features_df["lessons_viewed_raw"].notna().sum()),
            int(direct_view_features_df["lesson_media_sessions"].notna().sum()),
            int(direct_view_features_df["group_media_sessions"].notna().sum()),
        ],
    }
)
coverage_summary_df["share_pct"] = round(coverage_summary_df["rows"] / direct_view_features_df.shape[0] * 100, 2)

display(coverage_summary_df)


,metric,rows,share_pct
0,rows,4946,100.00
1,users_course_id_found,4927,99.62
2,user_lessons_found,4409,89.14
3,lesson_media_found,0,0.00
4,group_media_found,4323,87.40


По покрытию видно, что технический мост до `users_course_id` почти полный: `99.62%` строк. `user_lessons` тоже покрывают большую часть выборки (`89.14%`), а `group_media` находятся для `87.4%` строк. При этом `lesson_media` не находятся совсем: для этих модулей прямой сигнал просмотров идёт не через `resource_type = Lesson`, а почти целиком через `Group` и через агрегаты `user_lessons`.


## Сравнение прямых raw-сигналов с таргетом

Здесь пока не строим модель. Сначала смотрим, разделяют ли самые прямые сигналы просмотров классы `target = 1` и `target = 0`. Для этого сравниваем долю просмотренных уроков, количество просмотренных сегментов и две грубые бинарные прокси: `raw_lessons_80_status` и `raw_media_720_status`.


In [ ]:
raw_signal_summary_df = direct_view_features_df.groupby(["module", "metric_name", "target"], dropna=False).agg(
    rows=("user_id", "size"),
    lessons_viewed_share_mean=("lessons_viewed_share_raw", "mean"),
    lessons_viewed_share_median=("lessons_viewed_share_raw", "median"),
    total_media_segments_viewed_mean=("total_media_segments_viewed", "mean"),
    total_media_segments_viewed_median=("total_media_segments_viewed", "median"),
    lesson_media_share_mean=("lesson_media_share_raw", "mean"),
    group_media_share_mean=("group_media_share_raw", "mean"),
    raw_lessons_80_rate=("raw_lessons_80_status", "mean"),
    raw_media_720_rate=("raw_media_720_status", "mean"),
).reset_index()

raw_signal_summary_df[[
    "lessons_viewed_share_mean",
    "lessons_viewed_share_median",
    "total_media_segments_viewed_mean",
    "total_media_segments_viewed_median",
    "lesson_media_share_mean",
    "group_media_share_mean",
    "raw_lessons_80_rate",
    "raw_media_720_rate",
]] = raw_signal_summary_df[[
    "lessons_viewed_share_mean",
    "lessons_viewed_share_median",
    "total_media_segments_viewed_mean",
    "total_media_segments_viewed_median",
    "lesson_media_share_mean",
    "group_media_share_mean",
    "raw_lessons_80_rate",
    "raw_media_720_rate",
]].round(3)

display(raw_signal_summary_df)


,module,metric_name,target,rows,lessons_viewed_share_mean,lessons_viewed_share_median,total_media_segments_viewed_mean,total_media_segments_viewed_median,lesson_media_share_mean,group_media_share_mean,raw_lessons_80_rate,raw_media_720_rate
0,1,Просмотрено 80% ур или видеоконт,0,1019,0.294,0.174,70.013,0.0,NaN,0.296,0.064,0.007
1,1,Просмотрено 80% ур или видеоконт,1,1972,0.867,0.87,866.188,871.5,NaN,0.619,0.984,0.995
2,2,Просмотрено 720ед видеоконт и 80% ур,0,122,0.336,0.261,155.238,19.0,NaN,0.536,0.074,0.0
3,2,Просмотрено 720ед видеоконт и 80% ур,1,1833,0.864,0.87,866.284,880.0,NaN,0.763,0.96,0.999


Прямые raw-сигналы уже очень сильно разделяют классы. В обоих модулях у `target = 1` средняя доля просмотренных уроков около `0.86`, а у `target = 0` — около `0.29–0.34`. Для `module = 2` грубая бинарная прокси `raw_media_720_status` почти полностью совпадает с таргетом: доля выполнения равна `0.999` при `target = 1` и `0.000` при `target = 0`. Для `module = 1` аналогично сильной выглядит `raw_lessons_80_status`: `0.984` против `0.064`. Значит, метрики просмотров уже в значительной степени восстанавливаются прямыми признаками просмотра, и следующий шаг — понять, где именно остаются расхождения и какие дополнительные признаки помогают их объяснить.


## Где прямые raw-прокси ошибаются

Теперь отдельно смотрим на строки, где самая прямая прокси расходится с таргетом. Для `module = 1` такой прокси считаем `raw_lessons_80_status`, для `module = 2` — `raw_media_720_status`. Это позволит понять, какие случаи не объясняются простым правилом просмотра и требуют дополнительных признаков.


In [ ]:
direct_view_features_df["raw_proxy_status"] = pd.Series(
    np.where(
        direct_view_features_df["module"] == 1,
        direct_view_features_df["raw_lessons_80_status"],
        direct_view_features_df["raw_media_720_status"],
    ),
    index=direct_view_features_df.index,
    dtype="Int8",
)

mismatch_df = direct_view_features_df[
    direct_view_features_df["raw_proxy_status"] != direct_view_features_df["target"]
].copy()

module_1_rows = direct_view_features_df[direct_view_features_df["module"] == 1].shape[0]
module_2_rows = direct_view_features_df[direct_view_features_df["module"] == 2].shape[0]
module_1_mismatch_rows = mismatch_df[mismatch_df["module"] == 1].shape[0]
module_2_mismatch_rows = mismatch_df[mismatch_df["module"] == 2].shape[0]
all_rows = direct_view_features_df.shape[0]
all_mismatch_rows = mismatch_df.shape[0]

mismatch_summary_df = pd.DataFrame(
    {
        "module": [1, 2, "all"],
        "metric_name": [
            "Просмотрено 80% ур или видеоконт",
            "Просмотрено 720ед видеоконт и 80% ур",
            "both",
        ],
        "rows": [module_1_rows, module_2_rows, all_rows],
        "mismatch_rows": [module_1_mismatch_rows, module_2_mismatch_rows, all_mismatch_rows],
    }
)

mismatch_summary_df["mismatch_pct"] = (
    mismatch_summary_df["mismatch_rows"] / mismatch_summary_df["rows"] * 100
).round(2)

mismatch_example_df = mismatch_df[
    [
        "module",
        "metric_name",
        "user_id",
        "course_id",
        "target",
        "raw_proxy_status",
        "lessons_viewed_share_raw",
        "total_media_segments_viewed",
        "group_media_share_raw",
        "lessons_viewed_raw",
        "total_lessons",
    ]
].copy()

display(mismatch_summary_df)
display(mismatch_example_df.head(20))


TypeError: data type 'Int8' not understood

По этой таблице уже можно будет увидеть, насколько задача вообще сводится к простому просмотровому правилу. Если ошибок почти нет, значит метрика почти детерминирована прямыми raw-сигналами. Если ошибки заметны, следующий шаг — разбирать их по типам: поздний старт, короткое окно доступа, несоответствие между `user_lessons` и `wk_media_view_sessions`, особенности отдельных курсов или групп.


По mismatch-таблице уже видно, что почти все ошибки сидят в `module = 1`: `97` строк из `2991` (`3.24%`). В `module = 2` ошибок всего `2` из `1955` (`0.1%`). Это очень сильный результат: метрика `Просмотрено 720ед видеоконт и 80% ур` почти полностью восстанавливается прямой raw-прокси просмотров. Основная сложность остаётся в более мягкой метрике `module = 1`, где встречаются оба типа расхождений: и `target = 0, raw_proxy = 1`, и `target = 1, raw_proxy = 0`. Следующий разумный шаг — разбирать именно `module = 1` и проверять, помогают ли дополнительные признаки по времени старта, активности на курсе и структуре уроков объяснить эти `97` строк.


## Простой baseline без ML

Поскольку прямые raw-сигналы уже очень сильные, сначала проверяем самый простой baseline без модели: можно ли почти восстановить таргет одним порогом по `total_media_segments_viewed`. Это полезно и как sanity check, и как ориентир для будущей модели.


In [ ]:
module_1_df = direct_view_features_df[direct_view_features_df["module"] == 1].copy()
module_2_df = direct_view_features_df[direct_view_features_df["module"] == 2].copy()

module_1_thresholds_df = pd.DataFrame({"threshold": range(0, 1201, 10)})
module_1_thresholds_df["mismatch_rows"] = [
    int(((module_1_df["total_media_segments_viewed"] >= threshold).astype("Int8") != module_1_df["target"]).sum())
    for threshold in module_1_thresholds_df["threshold"]
]
module_1_thresholds_df["accuracy"] = (
    1 - module_1_thresholds_df["mismatch_rows"] / module_1_df.shape[0]
).round(4)

module_2_thresholds_df = pd.DataFrame({"threshold": range(0, 1201, 10)})
module_2_thresholds_df["mismatch_rows"] = [
    int(((module_2_df["total_media_segments_viewed"] >= threshold).astype("Int8") != module_2_df["target"]).sum())
    for threshold in module_2_thresholds_df["threshold"]
]
module_2_thresholds_df["accuracy"] = (
    1 - module_2_thresholds_df["mismatch_rows"] / module_2_df.shape[0]
).round(4)

best_module_1_row = module_1_thresholds_df.sort_values(["mismatch_rows", "threshold"]).iloc[0]
best_module_2_row = module_2_thresholds_df.sort_values(["mismatch_rows", "threshold"]).iloc[0]

best_threshold_summary_df = pd.DataFrame(
    {
        "module": [1, 2],
        "best_threshold": [
            int(best_module_1_row["threshold"]),
            int(best_module_2_row["threshold"]),
        ],
        "mismatch_rows": [
            int(best_module_1_row["mismatch_rows"]),
            int(best_module_2_row["mismatch_rows"]),
        ],
        "accuracy": [
            float(best_module_1_row["accuracy"]),
            float(best_module_2_row["accuracy"]),
        ],
    }
)

display(best_threshold_summary_df)
display(module_1_thresholds_df.sort_values(["mismatch_rows", "threshold"]).head(10))
display(module_2_thresholds_df.sort_values(["mismatch_rows", "threshold"]).head(10))


Подбор порога даёт сильный и немного неожиданный результат: и для `module = 1`, и для `module = 2` лучший простой threshold лежит примерно в зоне `680–700` просмотренных сегментов. Для `module = 1` это даёт всего `7` ошибок, для `module = 2` — `1` ошибку. Это очень сильный аргумент в пользу того, что обе метрики просмотров в реальности почти напрямую определяются одним и тем же raw-сигналом из `wk_media_view_sessions`.


## Почему ошибается прокси в `module = 1`

Теперь не добавляем сложные признаки, а просто разбираем сами `97` ошибок в `module = 1`. Цель здесь узкая: понять, это действительно сложные исключения или в основном эффект того, что прокси по `user_lessons` слабее, чем сигнал по `wk_media_view_sessions`.


In [24]:
module_1_mismatch_detail_df = mismatch_df[mismatch_df["module"] == 1].copy()
module_1_mismatch_detail_df["error_type"] = np.where(
    (module_1_mismatch_detail_df["target"] == 1)
    & (module_1_mismatch_detail_df["raw_proxy_status"] == 0),
    "target_1_proxy_0",
    "target_0_proxy_1",
)

module_1_mismatch_detail_df["required_lessons_for_80pct"] = np.ceil(
    module_1_mismatch_detail_df["total_lessons"] * 0.8
).astype("Int16")
module_1_mismatch_detail_df["one_lesson_short"] = (
    module_1_mismatch_detail_df["lessons_viewed_raw"]
    == module_1_mismatch_detail_df["required_lessons_for_80pct"] - 1
)

best_module_1_threshold = int(best_module_1_row["threshold"])
module_1_mismatch_detail_df["best_media_threshold_status"] = (
    module_1_mismatch_detail_df["total_media_segments_viewed"] >= best_module_1_threshold
).astype("Int8")
module_1_mismatch_detail_df["fixed_by_best_media_threshold"] = (
    module_1_mismatch_detail_df["best_media_threshold_status"]
    == module_1_mismatch_detail_df["target"]
)

module_1_mismatch_type_summary_df = module_1_mismatch_detail_df.groupby("error_type", dropna=False).agg(
    rows=("user_id", "size"),
    lessons_viewed_share_mean=("lessons_viewed_share_raw", "mean"),
    lessons_viewed_share_median=("lessons_viewed_share_raw", "median"),
    total_media_segments_viewed_mean=("total_media_segments_viewed", "mean"),
    total_media_segments_viewed_median=("total_media_segments_viewed", "median"),
    one_lesson_short_rate=("one_lesson_short", "mean"),
    media_720_or_more_rate=("raw_media_720_status", "mean"),
    fixed_by_best_media_threshold_rate=("fixed_by_best_media_threshold", "mean"),
).reset_index()

module_1_mismatch_type_summary_df[[
    "lessons_viewed_share_mean",
    "lessons_viewed_share_median",
    "total_media_segments_viewed_mean",
    "total_media_segments_viewed_median",
    "one_lesson_short_rate",
    "media_720_or_more_rate",
    "fixed_by_best_media_threshold_rate",
]] = module_1_mismatch_type_summary_df[[
    "lessons_viewed_share_mean",
    "lessons_viewed_share_median",
    "total_media_segments_viewed_mean",
    "total_media_segments_viewed_median",
    "one_lesson_short_rate",
    "media_720_or_more_rate",
    "fixed_by_best_media_threshold_rate",
]].round(3)

module_1_mismatch_examples_df = module_1_mismatch_detail_df[
    [
        "error_type",
        "user_id",
        "course_id",
        "target",
        "raw_proxy_status",
        "lessons_viewed_raw",
        "required_lessons_for_80pct",
        "lessons_viewed_share_raw",
        "total_media_segments_viewed",
        "best_media_threshold_status",
    ]
].sort_values(["error_type", "total_media_segments_viewed", "lessons_viewed_share_raw"], ascending=[True, False, False]).copy()

display(module_1_mismatch_type_summary_df)
display(module_1_mismatch_examples_df.head(20))


,error_type,rows,lessons_viewed_share_mean,lessons_viewed_share_median,total_media_segments_viewed_mean,total_media_segments_viewed_median,one_lesson_short_rate,media_720_or_more_rate,fixed_by_best_media_threshold_rate
0,target_0_proxy_1,65,0.866,0.87,357.569,224.0,0.0,0.092,0.908
1,target_1_proxy_0,32,0.774,0.783,788.469,799.5,0.812,0.969,1.0


,error_type,user_id,course_id,target,raw_proxy_status,lessons_viewed_raw,required_lessons_for_80pct,lessons_viewed_share_raw,total_media_segments_viewed,best_media_threshold_status
1768,target_0_proxy_1,711963,1029,0,1,20,19,0.869565,1522.0,1
1811,target_0_proxy_1,712137,1029,0,1,20,19,0.869565,1493.0,1
1460,target_0_proxy_1,708815,1029,0,1,20,19,0.869565,1456.0,1
1299,target_0_proxy_1,705191,1029,0,1,20,19,0.869565,1392.0,1
1494,target_0_proxy_1,708212,1029,0,1,20,19,0.869565,1383.0,1
1479,target_0_proxy_1,713129,1029,0,1,20,19,0.869565,1366.0,1
838,target_0_proxy_1,706332,1030,0,1,20,19,0.869565,679.0,0
658,target_0_proxy_1,703691,1029,0,1,20,19,0.869565,676.0,0
1591,target_0_proxy_1,715951,1030,0,1,19,19,0.826087,656.0,0
2217,target_0_proxy_1,721975,1029,0,1,20,19,0.869565,648.0,0


Разбор показывает, что ошибки в `module = 1` в основном не выглядят как сложные исключения. В строках `target = 1, raw_proxy = 0` почти все случаи уже имеют высокий медиасигнал: `96.9%` набирают хотя бы `720` сегментов, а `81.2%` вообще находятся всего в одном уроке от порога `80%`. В строках `target = 0, raw_proxy = 1` картина обратная: уроки формально покрыты, но медиапросмотр в среднем низкий.

Главный практический вывод здесь простой: проблема в `module = 1` сидит не столько в редких edge-case'ах, сколько в том, что `raw_lessons_80_status` как прокси слабее, чем медиасигнал. Лучший простой медиапорог исправляет `91` из `97` текущих ошибок, так что если дальше строить `model_df`, то базовым сильным признаком должен быть именно `total_media_segments_viewed`, а признаки по `user_lessons` стоит использовать скорее как дополнительный контекст.


## Итог

На этом этапе уже можно сделать два рабочих вывода. Во-первых, `module = 2` почти не требует сложного моделирования: целевая метрика практически восстанавливается напрямую из raw-просмотров. Во-вторых, даже для `module = 1` лучшая простая просмотровая прокси даёт очень мало ошибок. Значит, если следующая цель именно предсказание этих метрик, то основной упор стоит делать не на сложные модели, а на аккуратную реконструкцию просмотровых признаков и на разбор редких исключений, где raw-сигнал и таргет расходятся.


## Явный `raw_proxy` из сырых таблиц

Ниже ещё раз собираем `raw_proxy` отдельно для `module_1` и `module_2`, но теперь в максимально явном виде: берём только исходные raw-таблицы, вытаскиваем только нужные сырые поля, агрегируем их до уровня `user-course` и затем джойним фактический `target` из `module_1_metric_df` и `module_2_metric_df`.

Какие исходные признаки используются:

- `users_courses.id`, `users_courses.user_id`, `users_courses.course_id` — мост от пары `(user_id, course_id)` к `users_course_id`.
- `user_lessons.users_course_id`, `user_lessons.lesson_id`, `user_lessons.video_visited`, `user_lessons.translation_visited` — сырой сигнал того, что пользователь реально затронул урок.
- `lessons.id`, `lessons.course_id` — число уроков в курсе и привязка урока к курсу.
- `groups.id`, `groups.lesson_id` — мост от записи вебинара / группы к уроку.
- `wk_media_view_sessions.resource_type`, `wk_media_view_sessions.resource_id`, `wk_media_view_sessions.viewer_id`, `wk_media_view_sessions.segments_total`, `wk_media_view_sessions.viewed_segments_count` — сырой сигнал просмотров видеоконтента.
- `module_1_metric_df.user_id`, `module_1_metric_df.course_id`, `module_1_metric_df.group_template_id`, `module_1_metric_df.target` и аналогично те же поля из `module_2_metric_df` — фактический таргет для финального сравнения.

Логика эвристики такая:

1. Считаем `lessons_viewed_raw` как число строк `user_lessons`, где `video_visited == True` или `translation_visited == True`.
2. Считаем `lessons_viewed_share_raw = lessons_viewed_raw / total_lessons`.
3. Из `wk_media_view_sessions` оставляем только `resource_type == "Group"`, мапим `group -> lesson -> course` и суммируем `viewed_segments_count` в `total_media_segments_viewed`.
4. Финальный `raw_proxy` берём как самый сильный простой медиасигнал по модулю: `total_media_segments_viewed >= best_module_1_threshold` для `module_1` и `total_media_segments_viewed >= best_module_2_threshold` для `module_2`.

Здесь `raw_lessons_80_status` тоже сохраняем для прозрачности, но в финальный `raw_proxy` не включаем, потому что медиасигнал заметно лучше совпадает с фактическим таргетом.


In [21]:
raw_users_courses_bridge_df = users_courses_df[["id", "user_id", "course_id"]].rename(
    columns={"id": "users_course_id"}
).copy()

raw_course_lessons_df = lessons_df[["id", "course_id"]].copy()
raw_course_total_lessons_df = raw_course_lessons_df.groupby("course_id", dropna=False).agg(
    total_lessons=("id", "nunique")
).reset_index()

raw_user_lessons_base_df = user_lessons_df[
    ["users_course_id", "lesson_id", "video_visited", "translation_visited"]
].copy()
raw_user_lessons_base_df["lesson_viewed_raw"] = (
    raw_user_lessons_base_df["video_visited"].fillna(False)
    | raw_user_lessons_base_df["translation_visited"].fillna(False)
)
raw_user_lessons_agg_df = raw_user_lessons_base_df.groupby("users_course_id", dropna=False).agg(
    lessons_viewed_raw=("lesson_viewed_raw", "sum")
).reset_index()

raw_group_bridge_df = groups_df[["id", "lesson_id"]].rename(columns={"id": "group_id"}).copy()
raw_group_media_base_df = wk_media_view_sessions_df.loc[
    wk_media_view_sessions_df["resource_type"] == "Group",
    ["resource_type", "resource_id", "viewer_id", "segments_total", "viewed_segments_count"],
].copy()
raw_group_media_base_df = raw_group_media_base_df.rename(
    columns={"resource_id": "group_id", "viewer_id": "user_id"}
)
raw_group_media_base_df = raw_group_media_base_df.merge(raw_group_bridge_df, on="group_id", how="left")
raw_group_media_base_df = raw_group_media_base_df.merge(
    raw_course_lessons_df.rename(columns={"id": "lesson_id"}),
    on="lesson_id",
    how="left",
)
raw_group_media_agg_df = raw_group_media_base_df.groupby(["user_id", "course_id"], dropna=False).agg(
    total_media_segments_total=("segments_total", "sum"),
    total_media_segments_viewed=("viewed_segments_count", "sum"),
).reset_index()

best_module_1_threshold = int(best_module_1_row["threshold"])
best_module_2_threshold = int(best_module_2_row["threshold"])

def build_module_raw_proxy_df(module_target_df, metric_name, media_threshold):
    target_df = module_target_df[
        ["user_id", "course_id", "group_template_id", "target"]
    ].drop_duplicates(subset=["user_id", "course_id"]).copy()
    target_df["metric_name"] = metric_name

    df = target_df.merge(raw_users_courses_bridge_df, on=["user_id", "course_id"], how="left")
    df = df.merge(raw_course_total_lessons_df, on="course_id", how="left")
    df = df.merge(raw_user_lessons_agg_df, on="users_course_id", how="left")
    df = df.merge(raw_group_media_agg_df, on=["user_id", "course_id"], how="left")

    df["lessons_viewed_raw"] = df["lessons_viewed_raw"].fillna(0)
    df["total_media_segments_total"] = df["total_media_segments_total"].fillna(0)
    df["total_media_segments_viewed"] = df["total_media_segments_viewed"].fillna(0)
    df["media_threshold"] = media_threshold

    df["lessons_viewed_share_raw"] = df["lessons_viewed_raw"] / df["total_lessons"]
    df["raw_lessons_80_status"] = (df["lessons_viewed_share_raw"] >= 0.8).fillna(False).astype("Int8")
    df["raw_media_threshold_status"] = (
        df["total_media_segments_viewed"] >= media_threshold
    ).fillna(False).astype("Int8")

    df["raw_proxy"] = df["raw_media_threshold_status"]
    df["raw_proxy_match"] = df["raw_proxy"] == df["target"]
    df["lessons_viewed_share_raw"] = df["lessons_viewed_share_raw"].round(3)

    return df[
        [
            "metric_name",
            "user_id",
            "course_id",
            "group_template_id",
            "users_course_id",
            "total_lessons",
            "lessons_viewed_raw",
            "lessons_viewed_share_raw",
            "total_media_segments_total",
            "total_media_segments_viewed",
            "media_threshold",
            "raw_lessons_80_status",
            "raw_media_threshold_status",
            "raw_proxy",
            "target",
            "raw_proxy_match",
        ]
    ].copy()

module_1_raw_proxy_df = build_module_raw_proxy_df(
    module_1_metric_df,
    "Просмотрено 80% ур или видеоконт",
    best_module_1_threshold,
)
module_2_raw_proxy_df = build_module_raw_proxy_df(
    module_2_metric_df,
    "Просмотрено 720ед видеоконт и 80% ур",
    best_module_2_threshold,
)

raw_proxy_rebuild_summary_df = pd.DataFrame(
    {
        "module": [1, 2],
        "metric_name": [
            "Просмотрено 80% ур или видеоконт",
            "Просмотрено 720ед видеоконт и 80% ур",
        ],
        "media_threshold": [
            best_module_1_threshold,
            best_module_2_threshold,
        ],
        "rows": [
            module_1_raw_proxy_df.shape[0],
            module_2_raw_proxy_df.shape[0],
        ],
        "raw_proxy_1_rows": [
            int(module_1_raw_proxy_df["raw_proxy"].sum()),
            int(module_2_raw_proxy_df["raw_proxy"].sum()),
        ],
        "target_1_rows": [
            int(module_1_raw_proxy_df["target"].sum()),
            int(module_2_raw_proxy_df["target"].sum()),
        ],
        "matches": [
            int(module_1_raw_proxy_df["raw_proxy_match"].sum()),
            int(module_2_raw_proxy_df["raw_proxy_match"].sum()),
        ],
        "mismatch_rows": [
            int((~module_1_raw_proxy_df["raw_proxy_match"]).sum()),
            int((~module_2_raw_proxy_df["raw_proxy_match"]).sum()),
        ],
    }
)
raw_proxy_rebuild_summary_df["accuracy"] = (
    raw_proxy_rebuild_summary_df["matches"] / raw_proxy_rebuild_summary_df["rows"]
).round(4)

display(raw_proxy_rebuild_summary_df)
display(module_1_raw_proxy_df.head(15))
display(module_2_raw_proxy_df.head(15))


,module,metric_name,media_threshold,rows,raw_proxy_1_rows,target_1_rows,matches,mismatch_rows,accuracy
0,1,Просмотрено 80% ур или видеоконт,680,2991,1979,1972,2984,7,0.9977
1,2,Просмотрено 720ед видеоконт и 80% ур,680,1955,1832,1833,1954,1,0.9995


,metric_name,user_id,course_id,group_template_id,users_course_id,total_lessons,lessons_viewed_raw,lessons_viewed_share_raw,total_media_segments_total,total_media_segments_viewed,media_threshold,raw_lessons_80_status,raw_media_threshold_status,raw_proxy,target,raw_proxy_match
0,Просмотрено 80% ур или видеоконт,701741,1029,1149,588595,23.0,20,0.87,1029.0,894.0,680,1,1,1,1,True
1,Просмотрено 80% ур или видеоконт,737977,1029,1216,623160,23.0,20,0.87,1378.0,900.0,680,1,1,1,1,True
2,Просмотрено 80% ур или видеоконт,722851,1030,1186,602174,23.0,0,0.0,0.0,0.0,680,0,0,0,0,True
3,Просмотрено 80% ур или видеоконт,709724,1030,1108,573746,23.0,20,0.87,1725.0,979.0,680,1,1,1,1,True
4,Просмотрено 80% ур или видеоконт,717763,1030,1055,568668,23.0,0,0.0,0.0,0.0,680,0,0,0,0,True
5,Просмотрено 80% ур или видеоконт,702581,1030,1153,589585,23.0,20,0.87,1102.0,917.0,680,1,1,1,1,True
6,Просмотрено 80% ур или видеоконт,701022,1029,1151,589468,23.0,0,0.0,0.0,0.0,680,0,0,0,0,True
7,Просмотрено 80% ур или видеоконт,717748,1030,1186,602166,23.0,0,0.0,0.0,0.0,680,0,0,0,0,True
8,Просмотрено 80% ур или видеоконт,701139,1029,1152,589524,23.0,20,0.87,1393.0,905.0,680,1,1,1,1,True
9,Просмотрено 80% ур или видеоконт,729492,1029,1215,623066,23.0,20,0.87,999.0,881.0,680,1,1,1,1,True


,metric_name,user_id,course_id,group_template_id,users_course_id,total_lessons,lessons_viewed_raw,lessons_viewed_share_raw,total_media_segments_total,total_media_segments_viewed,media_threshold,raw_lessons_80_status,raw_media_threshold_status,raw_proxy,target,raw_proxy_match
0,Просмотрено 720ед видеоконт и 80% ур,701741,951,1258,714507,23,18,0.783,803.0,802.0,680,0,1,1,1,True
1,Просмотрено 720ед видеоконт и 80% ур,737977,951,1242,715020,23,20,0.87,1691.0,857.0,680,1,1,1,1,True
2,Просмотрено 720ед видеоконт и 80% ур,709724,954,1231,717009,23,19,0.826,1025.0,871.0,680,1,1,1,1,True
3,Просмотрено 720ед видеоконт и 80% ур,702581,954,965,714538,23,20,0.87,1084.0,903.0,680,1,1,1,1,True
4,Просмотрено 720ед видеоконт и 80% ур,701139,951,1295,720578,23,20,0.87,1186.0,948.0,680,1,1,1,1,True
5,Просмотрено 720ед видеоконт и 80% ур,729492,951,1239,715072,23,9,0.391,414.0,75.0,680,0,0,0,0,True
6,Просмотрено 720ед видеоконт и 80% ур,733054,951,1242,715028,23,20,0.87,1231.0,913.0,680,1,1,1,1,True
7,Просмотрено 720ед видеоконт и 80% ур,729760,951,1237,715080,23,20,0.87,1505.0,848.0,680,1,1,1,1,True
8,Просмотрено 720ед видеоконт и 80% ур,702459,954,1228,723498,23,20,0.87,902.0,773.0,680,1,1,1,1,True
9,Просмотрено 720ед видеоконт и 80% ур,706777,954,1275,717073,23,20,0.87,1617.0,1011.0,680,1,1,1,1,True


Этот явный пересчёт подтверждает основной вывод ноутбука в ещё более прикладном виде. Если брать только минимальный набор raw-полей и считать `raw_proxy` простым порогом по `total_media_segments_viewed`, то для `module_1` получается `2984` совпадения из `2991` (`99.77%`), а для `module_2` — `1954` из `1955` (`99.95%`).

Практически это значит, что в конце можно оставить две рабочие таблицы — `module_1_raw_proxy_df` и `module_2_raw_proxy_df` — как самую прямую реконструкцию просмотровой метрики из raw LMS-данных. В них уже лежат и сырые признаки, и рассчитанный `raw_proxy`, и фактический `target` для прямого сравнения.
